In [2]:
from complete_pipeline import CompletePipeline

pipeline = CompletePipeline(
    csv_path='../semantic_clustering/dbbe_full.csv',
    output_dir='./results',
    num_perms=[128],
    shingle_sizes=[3, 4],
    thresholds=[0.75, 0.8, 0.85],
    poem_thresholds=[0.2, 0.3, 0.4, 0.5],
    n_jobs=-1
)

pipeline.run_full_pipeline()

2025-11-27 15:18:52,085 - INFO - 
System detected: 32 cores, 376 GB RAM
2025-11-27 15:18:52,085 - INFO - → Workstation mode: Using 28 cores
2025-11-27 15:18:52,085 - INFO - → Large memory mode: 100000 chunk size
2025-11-27 15:18:52,086 - INFO - ======================================================================
2025-11-27 15:18:52,086 - INFO - COMPLETE PRODUCTION PIPELINE
2025-11-27 15:18:52,087 - INFO - ======================================================================
2025-11-27 15:18:52,087 - INFO - Scratch: /scratch/gent/vo/000/gvo00042/vsc48660/complete_clustering
2025-11-27 15:18:52,087 - INFO - Verse grid search: 1×2×3 = 6 configs
2025-11-27 15:18:52,087 - INFO - Poem grid search: 4 thresholds
2025-11-27 15:18:52,088 - INFO - 
Step 1: Loading and validating data
2025-11-27 15:18:52,366 - WARNING - Removing 7 empty verses
2025-11-27 15:18:52,374 - WARNING - Removing 12,739 verses with NaN ground truth
2025-11-27 15:18:52,380 - INFO - Loaded 41,424 verses from 10,821 poems


## Print sample results

In [3]:
import pandas as pd
import numpy as np
df=pd.read_csv('results/verse_clusters.csv')
df=df.dropna()
def show_example_verse_clusters(df: pd.DataFrame, n_clusters: int = 3):
    clusters = (
        df['predicted_verse_cluster']
        .dropna()
        .astype(int)
        .unique()
    )
    cluster_sizes = (
    df.groupby("predicted_verse_cluster")
      .size()
      .rename("count")
    )

    clusters_with_multiple = cluster_sizes[cluster_sizes > 1].index.tolist()
    example_clusters = np.random.choice(clusters_with_multiple, size=min(n_clusters, len(clusters)), replace=False)
    
    for cid in example_clusters:
        print(f"Cluster {cid}")
        
        subset = df[df['predicted_verse_cluster'] == cid]
        
        for _, row in subset.iterrows():
            print(f"Verse: {row['verse']}")
        
        print("\n")
show_example_verse_clusters(df, n_clusters=10)

Cluster 27180
Verse: χρησάμενος παῖ τῷ λογικῷ δικτύῳ:
Verse: Χρησάμ(εν)ος παῖ τῷ λογικῷ δικτύῳ,  


Cluster 3532
Verse: μᾶλλον δὲ τὸν γράψαντα Ἀββακοὺμ ταύτα
Verse: μᾶλλον δὲ τ(ὸν) γράψαντ(α) ἀββακοὺμ ταῦτα·


Cluster 1358
Verse: ἀπλανεο(ς) | σοφὰ μέτρα. θεηγορίης ἀναφαίνεις·
Verse: - ἀπλανεος σοφὰ μέτρα. θεηγο|ρίης ἀναφαίνεις·


Cluster 11519
Verse: δόλους τε πολλ(οὺς) κ(αὶ) πλοκὰς τῶν ψευσμ(ά)τ(ων) 
Verse: δόλ(ους) τε πολλ(οὺς) (καὶ) πλοκὰς τῶν ψευσμάτ(ων) 


Cluster 34006
Verse: πάσχοντας αἰσχρ(ῶς) ἐκ θε(ῶν) ὁμοτρόπ(ων) |
Verse: πάσχοντας αἰσχρῶς ἐκ θε(ῶν) ὁμοτρόπ(ων);


Cluster 35581
Verse: πιστῶν δὲ πᾶσαν πρὸς θεὸν πτεροῖ φρένα
Verse: πιστῶν δὲ πᾶσ(αν) πρὸς θ(εὸ)ν πτεροῖ φρένα:·
Verse: πιστ(ῶν) δὲ πᾶσαν πρὸς θ(εὸ)ν πτεροῖ φρένα:·
Verse: πιστῶν δὲ πᾶσαν πρὸς θ(εὸ)ν πτεροῖ φρένα:
Verse: πιστῶν δὲ πᾶσαν, πρὸ(ς) θ(εὸ)ν πτεροῖ φρέν(αν):
Verse: πιστῶν δὲ πᾶσαν, πρὸ(ς) θ(εὸ)ν πτεροῖ φρέν(αν):
Verse: πιστῶν δὲ πᾶσαν, πρὸ(ς) θ(εὸ)ν πτεροῖ φρέν(αν):
Verse: πιστῶν δὲ πᾶσαν πρὸς Θεὸν πτεροῖ φ

In [4]:
import pandas as pd

poems_df = pd.read_csv("results/poem_clusters.csv")
poems_df["idoriginal_poem"] = poems_df["idoriginal_poem"].astype(str)

cluster_sizes = poems_df.groupby("predicted_poem_cluster").size()
multi_poem_clusters = cluster_sizes[cluster_sizes > 1].index.tolist()
print(f"Found {len(multi_poem_clusters)} multi-poem clusters.")

clusters_to_show = multi_poem_clusters[:3]
poems_to_show = poems_df[poems_df["predicted_poem_cluster"].isin(clusters_to_show)]

cluster_to_poems = (
    poems_to_show.groupby("predicted_poem_cluster")["idoriginal_poem"]
    .apply(list)
    .to_dict()
)

verse_dict = {}  
chunksize = 100_000

for chunk in pd.read_csv("../concatenated.csv", chunksize=chunksize):
    chunk["idoriginal_poem"] = chunk["idoriginal_poem"].astype(str)
    chunk = chunk[chunk["idoriginal_poem"].isin(poems_to_show["idoriginal_poem"])]
    
    for poem_id, group in chunk.groupby("idoriginal_poem"):
        verses = group.sort_values("order")["verse"].tolist()
        verses = [str(v) for v in verses if pd.notna(v)]
        if poem_id in verse_dict:
            verse_dict[poem_id].extend(verses)
        else:
            verse_dict[poem_id] = verses

for cl in clusters_to_show:
    print("\n" + "="*80)
    print(f"CLUSTER {cl}")
    print("="*80)
    
    for poem_id in cluster_to_poems[cl]:
        poem_text = "\n".join(verse_dict.get(poem_id, []))
        print(f"\n--- Poem {poem_id} ---\n")
        print(poem_text)
        print("\n" + "-"*40)


Found 838 multi-poem clusters.

CLUSTER 9

--- Poem 16906 ---

ηρ εὐμαρίως ῥυέης με πάσης ἀλόγου λυμιαίης·
ελ ὕπτια ῥύβδην μισθάρνια πήματά μου·
ηρ σκεδάσας αὐτομάτως ἕωλα πόρειτ' ἀπρὶξ ἄρδην:
ελ τέλματα πάμπαν ἀφαίρων πόνεοντ' ἄκρω:
ηρ ῥύθμισον ὦ κατʹ ἀνώτατα πάντων πρύτανις ἄμμων:
ελ ἀλλʹ ἄνα τῆς δέ με παμφωτίας ἐλλογίμου:-
ηρ τευκτέα βίβλου νικόλαον δμῶα δ' ἀχρεῖον ὄντα:
ελ ἰσχάνας ἀμπλακίης ὄλβιον ἔργαζε τόν:
ηρ ὅλκεον εὐσύνοπτον πν(εύματο)ς σέθεν θείου:
ελ σκοπίαν ἐμφαίνων ἄρτιον ἀλκιτάτην.
[...]....[ -ca.?- ]
τὰ μὲν ἄλλα τω[ -ca.?- ]
οἱ δὲ τῶν σωμάτων .[ -ca.?- ]
εισι θαυμασιωτατοι[ -ca.?- ]
νη καὶ ἀξίωμα καὶ α.[ -ca.?- ]
της ἐστιν τῆς λαμπροτ.[ -ca.?- ]
ἀνατραφεὶς εὐνουστερ̣[ -ca.?- ]
ψαντες καὶ τῶν γνησίων του̣[ -ca.?- ]
μεν παρʼ ἡμεῖν(*) καὶ ἐντεῦθε̣[ν  -ca.?- ]
νῦν ἐπιφανῶς ἀχθέντ̣[ -ca.?- ]
πενταετηρικὸν πρῶτ̣ο̣ν̣ .[ -ca.?- ]
μον εἰσ̣ε̣λ̣α̣στικοῦ̣ των....[ -ca.?- ]
ἀγωνισάμενος ἐστεφανώθη ος...[ -ca.?- ]
ητησεν παρὰ τῶν ἀγωνοθετῶν ...[ -ca.?- ]
μεν αὐτὸν εἴδομεν ἀμείψασθαι .